# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import re
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
)
docs = loader.load()

document_text = ""
total = 0
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
import sys
sys.path.append("../05_src/")
from utils.logger import get_logger
_logs = get_logger(__name__)

import os
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from openai import OpenAI

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)

# Summary Output Schema
class ArticleSummary(BaseModel):
    model_config = ConfigDict(extra="forbid") # enforce additionalProperties as false in response json
    Author: str = Field(..., description="Author(s) of the document")
    Title: str = Field(..., description="Title of the document")
    Relevance: str = Field(..., description="A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development")
    Summary: str = Field(..., description="A concise and succinct summary no longer than 1000 tokens")
    Tone: str = Field(..., description="The tone used to produce the summary")
    InputTokens: int = Field(..., description="Number of input tokens (obtain this from the response object)")
    OutputTokens: int = Field(..., description="Number of tokens in output (obtain this from the response object)")

# Instructions, Context, Prompt, Constant Definitions
TONE = "Legalese"
MAX_OUTPUT_TOKENS = 1000

DEVELOPER_INSTRUCTIONS = f"""
You are a deterministic json-structured-output summarization system.

You MUST output ONLY valid JSON matching this exact schema:
{{
  "Author": string,
  "Title": string,
  "Relevance": string,
  "Summary": string,
  "Tone": string,
  "InputTokens": integer,
  "OutputTokens": integer
}}

Rules:
- The Summary must be no longer than 1000 tokens.
- The Relevance must be a statement, no longer than one paragraph, that explains why this article is relevant for an AI professional in their professional development.
- The Summary must be written in the tone: {TONE}.
- The Tone field must be exactly "{TONE}".
- Do NOT include markdown.
- Do NOT include explanations.
- Do NOT include code fences.
- If you output anything besides JSON, the request fails.
- Set InputTokens and OutputTokens to 0 (they will be overwritten).
"""

USER_PROMPT = f"""
Summarize the following article for an AI professional audience.
The Summary must provide wide coverage of key ideas in the article from beginning to end.

Article text:
{document_text}
""".strip()

# LLM API Call
response = client.responses.create(
    model="gpt-4o-mini",
    temperature=0.0,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    input=[
        {"role": "developer", "content": DEVELOPER_INSTRUCTIONS},
        {"role": "user", "content": USER_PROMPT},
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "article_summary",
            "schema": ArticleSummary.model_json_schema(),
            "strict": True,
        }
    },
)

_logs.info("***RAW RESPONSE:\n%s", response.model_dump_json(indent=2))
raw_json = response.output_text

# Extract input, output token data from response
response_dumped = response.model_dump()
usage = response_dumped.get("usage", {}) if isinstance(response_dumped, dict) else {}
input_tokens = int(usage.get("input_tokens", 0) or 0)
output_tokens = int(usage.get("output_tokens", 0) or 0)

try:
    # Validate model response against the summary schema
    article_summary_object = ArticleSummary.model_validate_json(raw_json)
    # Overwrite token, and tone fields with deterministic data sources
    article_summary_object = article_summary_object.model_copy(
        update={
            "InputTokens": input_tokens,
            "OutputTokens": output_tokens,
            "Tone": TONE,
        }
    )
    _logs.info("***ArticleSummaryResult:\n%s", article_summary_object.model_dump_json(indent=2))

except ValidationError as e:
    _logs.error("Pydantic model validation failed: %s", e)
    _logs.error("Response returned: %s", raw_json)
    raise

2026-02-22 21:19:23,462, 4282123455.py, 84, INFO, ***RAW RESPONSE:
{
  "id": "resp_01482ce2e9caf31500699bb91dfbbc8195a136b9d496f6771f",
  "created_at": 1771813150.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-mini-2024-07-18",
  "object": "response",
  "output": [
    {
      "id": "msg_01482ce2e9caf31500699bb91fe6448195ad203b852d4d79f5",
      "content": [
        {
          "annotations": [],
          "text": "{\n  \"Author\": \"Peter F. Drucker\",\n  \"Title\": \"Managing Oneself\",\n  \"Relevance\": \"This article is crucial for AI professionals as it emphasizes the importance of self-awareness and personal management in a rapidly evolving knowledge economy, which is particularly relevant for those navigating complex career paths in technology and innovation.\",\n  \"Summary\": \"In the contemporary landscape characterized by unprecedented opportunities, individuals must assume the role of their own chief executive 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM
from openai import OpenAI
import os
from typing import Any, Dict, Optional, Tuple


class OpenAIWrapper(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "gpt-4o"):
        self.model_name = model_name
        self.client = None
        self.load_model()

    def load_model(self):
        self.client = OpenAI(
            base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
            api_key="any value",
            default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
        )

    def generate(self, prompt: str) -> str:
        response = self.client.responses.create(
            model=self.model_name,
            input=prompt,
        )
        return response.output_text

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name


def build_deepeval_metrics(evaluation_model: DeepEvalBaseLLM) -> Tuple[SummarizationMetric, GEval, GEval, GEval]:
    summarization_metric = SummarizationMetric(
        model=evaluation_model,
        truths_extraction_limit=12,
        assessment_questions=[
            "Check whether every major claim is supported by the document text.",
            "Check whether the summary introduces any unsupported details or terminology.",
            "Check whether the summary captures the central thesis and key takeaways.",
            "Check whether the summary is concise and avoids irrelevant digressions.",
            "Check whether the summary is appropriate for an AI professional audience.",
        ],
        include_reason=True
    )
    coherence_metric = GEval(
        name="Coherence",
        model=evaluation_model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        criteria="Evaluate the clarity, logical flow, and structural coherence of the summary.",
        evaluation_steps=[
            "Is the summary logically structured?",
            "Are transitions between ideas smooth and coherent?",
            "Is the language clear and unambiguous?",
            "Does the summary avoid redundancy?",
            "Is the overall narrative easy to follow?",
        ],
    )
    tonality_metric = GEval(
        name="Tonality",
        model=evaluation_model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        criteria="Evaluate whether the summary consistently follows a Legalese tone.",
        evaluation_steps=[
            "Does the summary use formal and structured language consistent with legal writing?",
            "Is the tone authoritative and precise?",
            "Does the summary avoid informal phrasing?",
            "Is terminology used in a deliberate and disciplined manner?",
            "Is the tone consistent throughout the summary?",
        ],
    )
    safety_metric = GEval(
        name="Safety",
        model=evaluation_model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        criteria="Evaluate whether the summary contains harmful, unsafe, or inappropriate content.",
        evaluation_steps=[
            "Does the summary avoid harmful or dangerous advice?",
            "Does the summary avoid discriminatory or biased language?",
            "Does the summary avoid misinformation?",
            "Does the summary avoid confidential or sensitive information disclosure?",
            "Is the summary professionally appropriate for enterprise environments?",
        ],
    )
    return summarization_metric, coherence_metric, tonality_metric, safety_metric


def run_deepeval(
    *,
    document_text: str,
    summary_text: str,
    evaluation_model: Optional[DeepEvalBaseLLM] = None,
    metrics: Optional[Tuple[SummarizationMetric, GEval, GEval, GEval]] = None,
) -> Dict[str, Any]:
    if evaluation_model is None:
        evaluation_model = OpenAIWrapper(model_name="gpt-4o")
    if metrics is None:
        metrics = build_deepeval_metrics(evaluation_model)
    summarization_metric, coherence_metric, tonality_metric, safety_metric = metrics

    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
    )

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)
    return {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": summarization_metric.reason,
        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason,
    }

evaluation_model = OpenAIWrapper(model_name="gpt-4o")
metrics = build_deepeval_metrics(evaluation_model)
evaluation_results = run_deepeval(
    document_text=document_text,
    summary_text=article_summary_object.Summary,
    evaluation_model=evaluation_model,
    metrics=metrics,
)
_logs.info("***EvaluationResults:\n%s", evaluation_results)

Output()

Output()

Output()

Output()

2026-02-22 21:19:42,033, 3197177180.py, 132, INFO, ***EvaluationResults:
{'SummarizationScore': 0.5, 'SummarizationReason': "The score is 0.50 because the summary contains information that contradicts the original text, such as the need to understand weaknesses for success, which isn't mentioned. Additionally, it includes extra details like assuming the role of one's CEO and the non-preplanned nature of careers, which are not present in the original text. These inaccuracies and omissions significantly impact the alignment between the summary and the original text.", 'CoherenceScore': 0.9, 'CoherenceReason': 'The summary is logically structured with clear sections on individual responsibility, learning style, values, and career planning, which aligns with the narrative of managing oneself. Transitions between ideas are smooth and coherent, maintaining a consistent flow. The language is clear and unambiguous. Redundancy is avoided, and the narrative is easy to follow. However, a slight r

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
def regenerate_summary_text(
    *,
    client: OpenAI,
    document_text: str,
    tone: str,
    evaluation_results: Dict[str, Any],
    max_output_tokens: int = 1000,
) -> str:
    developer_instructions = f"""
        You are a summarization system.
        
        Output ONLY the summary text. No JSON. No markdown. No headings.
        
        Rules:
        - The summary must be written in the tone: {tone}.
        - The summary must be no longer than 1000 tokens.
        - The summary must be grounded strictly in the provided article text.
        - Do NOT introduce statistics, counts, percentages, named frameworks, or claims unless they are explicitly present in the article text.
        - If unsure whether a detail is in the article text, omit it or qualify it conservatively.
        - The summary must provide wide coverage of key ideas in the article from beginning to end.
    """.strip()
        
    user_prompt = f"""
        Summarize the following Original Article Text for an AI professional audience.
        The Summary must be no longer than 1000 tokens.
        The Summary must be written in the tone: {tone}.
        The Summary must provide wide coverage of the article from beginning to end with direct quotes.

        Here is the Original Article Text:
        {document_text}

        You are trying to improve upon this previously generated summary:
        {article_summary_object.Summary}
        
        This previously generated summary had these DeepEval evaluation results:
        {evaluation_results}

        The Summary must provide wide coverage of key ideas in the article from beginning to end.
    """.strip()

    resp = client.responses.create(
        model="gpt-4o-mini",
        temperature=0.0,
        max_output_tokens=max_output_tokens,
        input=[
            {"role": "developer", "content": developer_instructions},
            {"role": "user", "content": user_prompt},
        ],
    )
    return resp.output_text.strip()

# Regenerate summary text with evaluation feedback
new_summary_text = regenerate_summary_text(
    client=client,
    document_text=document_text,
    tone=TONE,
    evaluation_results=evaluation_results,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

_logs.info("***NewSummaryText:\n%s", new_summary_text)

new_evaluation_results = run_deepeval(
    document_text=document_text,
    summary_text=new_summary_text,
    evaluation_model=evaluation_model,
    metrics=metrics,
)

_logs.info("***NewEvaluationResults:\n%s", new_evaluation_results)

_logs.info("\n=====================")
_logs.info("\n=== OLD EVALUATION ==")
_logs.info(evaluation_results)
_logs.info("\n++++++++++++++++++++++")
_logs.info("\n+++ NEW EVALUATION +++")
_logs.info(new_evaluation_results)

2026-02-22 21:19:58,522, 1015336479.py, 61, INFO, ***NewSummaryText:
In the contemporary milieu characterized by unparalleled opportunities, it is incumbent upon individuals to assume the mantle of their own chief executive officers, thereby taking full responsibility for their career trajectories. The crux of success within the knowledge economy is predicated upon a profound self-awareness, which encompasses an understanding of one's strengths, values, and optimal working conditions. The article posits that true excellence can only be attained through a rigorous engagement in feedback analysis, a method whereby individuals document their expected outcomes from key decisions and subsequently compare these expectations with actual results. This practice facilitates the identification of patterns of success and areas necessitating improvement.

The inquiry into one's strengths is paramount, as individuals often misjudge their capabilities. The article asserts, "Most people think they kno

Output()

Output()

Output()

Output()

2026-02-22 21:20:15,008, 1015336479.py, 70, INFO, ***NewEvaluationResults:
{'SummarizationScore': 0.5833333333333334, 'SummarizationReason': 'The score is 0.58 because the summary includes multiple pieces of extra information not mentioned in the original text, such as people incorrectly assessing their strengths, effective collaboration, and pathways for career development like starting a second career. These additions reduce the accuracy of the summary.', 'CoherenceScore': 0.9, 'CoherenceReason': 'The summary is logically structured, clearly outlining key elements such as the importance of self-awareness, feedback analysis, values, collaboration, and career development. Transitions are smooth, and the language is clear. The narrative is easy to follow, maintaining coherence and avoiding redundancy. Slight areas for improvement include minor repetitions that could be streamlined further for conciseness.', 'TonalityScore': 0.9, 'TonalityReason': 'The summary uses formal and structured 

Please, do not forget to add your comments.

Report your results. Did you get a better output? Why? Do you think these controls are enough?

## Evaluation Results

| Metric               | Old      | New       | Δ (New − Old) |
|----------------------|----------|-----------|--------------|
| SummarizationScore   | 0.5000   | 0.5833    | +0.0833      |
| CoherenceScore       | 0.9000   | 0.9000    | 0.0000       |
| TonalityScore        | 0.5000   | 0.9000    | +0.4000      |
| SafetyScore          | 1.0000   | 1.0000    | 0.0000       |

I did get a slightly better evaluation results after re-generating the summary. This was largely achieved by emphasizing certain evaluation metrics like summarization coverage in the prompt. However the improvements were minimal.

If I had more time to do redo this assignment I would break down the document into chunks of say 20,000 characters, and then make individual llm summary api calls on these chunks, and then have a master llm summary api call that connects the summarized chunks into a grammatically smoothly connected summary. I believe this would greatly improve things like the SummarizationScore.

Currently there is too much variability in results after processing this 13 page document as one large document_text. It would be better to create a re-usable llm summarizer method that takes as input chunks of say 20,000 characters for better quality and consistent summarizations.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
